# 01. GA4 E-Commerce Data Cleaning & Preprocessing

This notebook performs data ingestion, exploratory quality checks, type casting, timestamp conversion, and cleaning on raw Google Analytics 4 (GA4) event export data.

In [13]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 1: Load Dataset

In [14]:
raw_data_path = '../data/raw/ga4_raw_data.csv'
df= pd.read_csv(raw_data_path)

print(f"Raw dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

Raw dataset shape: 50,000 rows, 10 columns


,event_date,formatted_date,event_timestamp,event_name,user_pseudo_id,device_category,traffic_source,traffic_medium,session_id,engagement_time_msec
0,20210103,2021-01-03,1609677048640714,page_view,2.051660e+06,desktop,(data deleted),(data deleted),2660893926,2.0
1,20210103,2021-01-03,1609677141092350,page_view,2.051660e+06,desktop,(data deleted),(data deleted),2660893926,2.0
2,20210103,2021-01-03,1609676901062582,user_engagement,2.051660e+06,desktop,(data deleted),(data deleted),2660893926,2595.0
3,20210103,2021-01-03,1609677154552035,page_view,2.051660e+06,desktop,(data deleted),(data deleted),2660893926,2.0
4,20210103,2021-01-03,1609676270905669,user_engagement,2.051660e+06,desktop,(data deleted),(data deleted),2660893926,15548.0


In [15]:
df.info()

df.describe(include="all")

df.dtypes

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   event_date            50000 non-null  int64  
 1   formatted_date        50000 non-null  str    
 2   event_timestamp       50000 non-null  int64  
 3   event_name            50000 non-null  str    
 4   user_pseudo_id        50000 non-null  float64
 5   device_category       50000 non-null  str    
 6   traffic_source        50000 non-null  str    
 7   traffic_medium        50000 non-null  str    
 8   session_id            50000 non-null  int64  
 9   engagement_time_msec  28298 non-null  float64
dtypes: float64(2), int64(3), str(5)
memory usage: 5.9 MB


event_date                int64
formatted_date              str
event_timestamp           int64
event_name                  str
user_pseudo_id          float64
device_category             str
traffic_source              str
traffic_medium              str
session_id                int64
engagement_time_msec    float64
dtype: object

### 2. Data Quality & Missing Value Assessment

In [16]:
missing_values = df.isnull().sum()

missing_percentage = (
    df.isnull().mean() * 100
)

missing_report = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": missing_percentage
})

print(missing_report)

                      Missing_Count  Missing_Percentage
event_date                        0               0.000
formatted_date                    0               0.000
event_timestamp                   0               0.000
event_name                        0               0.000
user_pseudo_id                    0               0.000
device_category                   0               0.000
traffic_source                    0               0.000
traffic_medium                    0               0.000
session_id                        0               0.000
engagement_time_msec          21702              43.404


In [17]:
#Duplicate Investigation
print("Total duplicates:", df.duplicated().sum())

Total duplicates: 0


### 4: Data Type Conversion

In [18]:
df["formatted_date"] = pd.to_datetime(
    df["formatted_date"],
    errors="coerce"
)

df["event_timestamp"] = pd.to_numeric(
    df["event_timestamp"],
    errors="coerce"
)

df["engagement_time_msec"] = pd.to_numeric(
    df["engagement_time_msec"],
    errors="coerce"
)

### 5. Handle Inconsistent Values

In [19]:
df.replace(
    ["(not set)", "(data deleted)", ""],
    np.nan,
    inplace=True
)

,event_date,formatted_date,event_timestamp,event_name,user_pseudo_id,device_category,traffic_source,traffic_medium,session_id,engagement_time_msec
0,20210103,2021-01-03,1609677048640714,page_view,2.051660e+06,desktop,NaN,NaN,2660893926,2.0
1,20210103,2021-01-03,1609677141092350,page_view,2.051660e+06,desktop,NaN,NaN,2660893926,2.0
2,20210103,2021-01-03,1609676901062582,user_engagement,2.051660e+06,desktop,NaN,NaN,2660893926,2595.0
3,20210103,2021-01-03,1609677154552035,page_view,2.051660e+06,desktop,NaN,NaN,2660893926,2.0
4,20210103,2021-01-03,1609676270905669,user_engagement,2.051660e+06,desktop,NaN,NaN,2660893926,15548.0
...,...,...,...,...,...,...,...,...,...,...
49995,20201110,2020-11-10,1605005944051997,page_view,3.062371e+07,desktop,shop.googlemerchandisestore.com,referral,9399354612,NaN
49996,20201110,2020-11-10,1605007217202928,page_view,3.062371e+07,desktop,shop.googlemerchandisestore.com,referral,9399354612,NaN
49997,20201110,2020-11-10,1605005767542455,page_view,3.062371e+07,desktop,shop.googlemerchandisestore.com,referral,9399354612,NaN
49998,20201110,2020-11-10,1605045540844205,scroll,3.135858e+07,mobile,shop.googlemerchandisestore.com,referral,3132793688,NaN


### 6. Export Cleaned Dataset

In [12]:
df.to_csv(
    "../data/processed/ga4_cleaned_data.csv",
    index=False
)